# Module 4 Lab — Agent Identity & Delegated Authority

**Scenario:** Enterprise Procurement Agent

This lab builds a concrete authority chain:

```text
Human identity
  ↓
Delegation
  ↓
Agent identity
  ↓
Workload identity
  ↓
Task-scoped authority
  ↓
Authorization check
  ↓
Tool execution
  ↓
Audit evidence
```

The core exercises run locally. OpenFGA is an optional live extension.

## Libraries and standards

- **Pydantic** — typed identity/delegation contracts
- **PyJWT + cryptography** — signed training delegation token
- **OpenFGA Python SDK** — optional live relationship/task authorization
- **pandas** — evidence inspection
- **RFC 8693** — OAuth Token Exchange concepts
- **RFC 9700** — OAuth security BCP
- **SPIFFE/SPIRE** — workload identity concepts
- **Cedar** — contextual authorization model

> The local JWT issuer is for teaching. In production, use an approved identity provider / authorization server / STS.

In [ ]:
%pip install -q "pydantic>=2" PyJWT cryptography pandas requests openfga_sdk
print("Dependencies installed.")

In [ ]:
from __future__ import annotations
from datetime import datetime, timezone, timedelta
from enum import Enum
from typing import Any, Optional
from uuid import uuid4
import os, json

import jwt
import pandas as pd
from pydantic import BaseModel, Field
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.hazmat.primitives import serialization

pd.set_option("display.max_colwidth", 120)

## 1. Model human, agent, and workload identities

In [ ]:
class PrincipalType(str, Enum):
    HUMAN = "human"
    AGENT = "agent"
    WORKLOAD = "workload"

class Principal(BaseModel):
    principal_id: str
    principal_type: PrincipalType
    display_name: str
    owner: Optional[str] = None
    version: Optional[str] = None

human = Principal(
    principal_id="human:user-123",
    principal_type=PrincipalType.HUMAN,
    display_name="Analytics Manager",
)

agent = Principal(
    principal_id="agent:procurement-v1",
    principal_type=PrincipalType.AGENT,
    display_name="Procurement Agent",
    owner="AI Platform",
    version="1.0.0",
)

workload = Principal(
    principal_id="spiffe://oneplusi.example/prod/procurement-agent",
    principal_type=PrincipalType.WORKLOAD,
    display_name="Procurement Agent workload",
)

display(pd.DataFrame([p.model_dump() for p in [human, agent, workload]]))

### Why three identities?

The human identifies the **delegator**.

The logical agent identifies the governed AI actor/version.

The workload identity identifies the **running software instance/environment**.

Do not collapse all three into one shared credential.

## 2. Define a task-scoped delegation grant

In [ ]:
class DelegationConstraints(BaseModel):
    max_amount: float = 0
    allowed_vendor_ids: list[str] = []
    max_calls: int = 1

class DelegationGrant(BaseModel):
    grant_id: str = Field(default_factory=lambda: f"grant-{uuid4().hex[:8]}")
    subject: str
    actor: str
    workload: str
    task_id: str
    audience: str
    permissions: list[str]
    resources: list[str]
    constraints: DelegationConstraints
    issued_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))
    expires_at: datetime
    parent_grant_id: Optional[str] = None

grant = DelegationGrant(
    subject=human.principal_id,
    actor=agent.principal_id,
    workload=workload.principal_id,
    task_id="task-buy-laptops-001",
    audience="procurement-api",
    permissions=["vendor:read", "purchase_order:create"],
    resources=["department:data-ai"],
    constraints=DelegationConstraints(
        max_amount=5000,
        allowed_vendor_ids=["vendor-acme", "vendor-northstar"],
        max_calls=1,
    ),
    expires_at=datetime.now(timezone.utc) + timedelta(minutes=30),
)

print(grant.model_dump_json(indent=2))

## 3. Create a local training STS keypair

In [ ]:
private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
public_key = private_key.public_key()

private_pem = private_key.private_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption(),
)
public_pem = public_key.public_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PublicFormat.SubjectPublicKeyInfo,
)

print(public_pem.decode().splitlines()[0], "...", public_pem.decode().splitlines()[-1])

## 4. Issue a short-lived signed delegation token

In [ ]:
def issue_grant_token(g: DelegationGrant) -> str:
    payload = {
        "iss": "https://training-sts.oneplusi.example",
        "sub": g.subject,
        "act": {"sub": g.actor},
        "workload": g.workload,
        "task": g.task_id,
        "aud": g.audience,
        "scope": " ".join(g.permissions),
        "resources": g.resources,
        "constraints": g.constraints.model_dump(),
        "grant_id": g.grant_id,
        "parent_grant_id": g.parent_grant_id,
        "iat": int(g.issued_at.timestamp()),
        "exp": int(g.expires_at.timestamp()),
        "jti": uuid4().hex,
    }
    return jwt.encode(payload, private_pem, algorithm="RS256")

token = issue_grant_token(grant)
print(token[:90] + "...")

The `act` idea mirrors RFC 8693's actor/subject delegation semantics.

This notebook is **not** an OAuth server implementation. It uses a signed local token to make the delegation mechanics visible.

## 5. Define an authorization request

In [ ]:
class AuthorizationRequest(BaseModel):
    actor: str
    workload: str
    action: str
    resource: str
    audience: str
    task_id: str
    amount: float = 0
    vendor_id: Optional[str] = None

safe_request = AuthorizationRequest(
    actor=agent.principal_id,
    workload=workload.principal_id,
    action="purchase_order:create",
    resource="department:data-ai",
    audience="procurement-api",
    task_id=grant.task_id,
    amount=4500,
    vendor_id="vendor-acme",
)

## 6. Validate token identity, audience, expiry, task, resource, and limits

In [ ]:
REVOKED_GRANTS = set()
CALL_COUNTS = {}

def decode_grant(token: str, audience: str) -> dict[str, Any]:
    return jwt.decode(
        token,
        public_pem,
        algorithms=["RS256"],
        audience=audience,
        issuer="https://training-sts.oneplusi.example",
    )

def authorize(token: str, request: AuthorizationRequest):
    try:
        claims = decode_grant(token, request.audience)
    except Exception as exc:
        return False, f"Token validation failed: {type(exc).__name__}", {}

    if claims["grant_id"] in REVOKED_GRANTS:
        return False, "Grant revoked", claims
    if claims["act"]["sub"] != request.actor:
        return False, "Actor mismatch", claims
    if claims["workload"] != request.workload:
        return False, "Workload mismatch", claims
    if claims["task"] != request.task_id:
        return False, "Task mismatch", claims
    if request.action not in claims["scope"].split():
        return False, "Action not delegated", claims
    if request.resource not in claims["resources"]:
        return False, "Resource outside delegated scope", claims

    constraints = claims["constraints"]
    if request.amount > constraints["max_amount"]:
        return False, "Amount exceeds delegated limit", claims

    allowed_vendors = constraints.get("allowed_vendor_ids", [])
    if request.vendor_id and allowed_vendors and request.vendor_id not in allowed_vendors:
        return False, "Vendor outside delegated scope", claims

    if CALL_COUNTS.get(claims["grant_id"], 0) >= constraints.get("max_calls", 1):
        return False, "Grant call limit exhausted", claims

    return True, "Authorized", claims

authorize(token, safe_request)[:2]

## 7. Put the check in a Policy Enforcement Point

In [ ]:
AUDIT = []

def create_purchase_order(token: str, request: AuthorizationRequest):
    allowed, reason, claims = authorize(token, request)

    event = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "grant_id": claims.get("grant_id"),
        "subject": claims.get("sub"),
        "actor": request.actor,
        "workload": request.workload,
        "task": request.task_id,
        "action": request.action,
        "resource": request.resource,
        "amount": request.amount,
        "vendor": request.vendor_id,
        "allowed": allowed,
        "reason": reason,
    }

    if not allowed:
        AUDIT.append(event)
        return event

    CALL_COUNTS[claims["grant_id"]] = CALL_COUNTS.get(claims["grant_id"], 0) + 1
    event["result"] = {"po_id": f"PO-{uuid4().hex[:6]}", "status": "created"}
    AUDIT.append(event)
    return event

create_purchase_order(token, safe_request)

## 8. Verify replay / call-limit protection

In [ ]:
# Same grant is limited to one execution.
create_purchase_order(token, safe_request)

## 9. Test over-scoped requests

In [ ]:
def fresh_grant():
    g = grant.model_copy(update={
        "grant_id": f"grant-{uuid4().hex[:8]}",
        "issued_at": datetime.now(timezone.utc),
        "expires_at": datetime.now(timezone.utc) + timedelta(minutes=30),
    })
    return g, issue_grant_token(g)

tests = [
    ("over amount", safe_request.model_copy(update={"amount": 12000})),
    ("wrong vendor", safe_request.model_copy(update={"vendor_id": "vendor-evil"})),
    ("wrong resource", safe_request.model_copy(update={"resource": "department:finance"})),
    ("wrong task", safe_request.model_copy(update={"task_id": "task-other"})),
    ("wrong workload", safe_request.model_copy(update={
        "workload": "spiffe://oneplusi.example/prod/other-agent"
    })),
]

rows = []
for name, req in tests:
    g, t = fresh_grant()
    ok, reason, _ = authorize(t, req)
    rows.append({"test": name, "allowed": ok, "reason": reason})

display(pd.DataFrame(rows))

## 10. Revoke authority

In [ ]:
revocable, revocable_token = fresh_grant()
print(authorize(revocable_token, safe_request)[:2])

REVOKED_GRANTS.add(revocable.grant_id)

print(authorize(revocable_token, safe_request)[:2])

## 11. Sub-agent delegation with authority attenuation

In [ ]:
research_agent = Principal(
    principal_id="agent:research-v1",
    principal_type=PrincipalType.AGENT,
    display_name="Vendor Research Agent",
    owner="AI Platform",
    version="1.0.0",
)

def attenuate_grant(
    parent: DelegationGrant,
    child_actor: str,
    permissions: list[str],
    max_calls: int = 3,
) -> DelegationGrant:
    if not set(permissions).issubset(set(parent.permissions)):
        raise ValueError("Child permissions cannot exceed parent permissions.")

    return DelegationGrant(
        subject=parent.subject,
        actor=child_actor,
        workload="spiffe://oneplusi.example/prod/research-agent",
        task_id=parent.task_id,
        audience="vendor-api",
        permissions=permissions,
        resources=parent.resources,
        constraints=DelegationConstraints(
            max_amount=0,
            allowed_vendor_ids=parent.constraints.allowed_vendor_ids,
            max_calls=max_calls,
        ),
        issued_at=datetime.now(timezone.utc),
        expires_at=min(
            parent.expires_at,
            datetime.now(timezone.utc) + timedelta(minutes=10),
        ),
        parent_grant_id=parent.grant_id,
    )

child_grant = attenuate_grant(grant, research_agent.principal_id, ["vendor:read"])
print(child_grant.model_dump_json(indent=2))

In [ ]:
try:
    attenuate_grant(grant, research_agent.principal_id, ["vendor:read", "payment:issue"])
except ValueError as exc:
    print("Blocked privilege amplification:", exc)

# 12. OAuth 2.0 Token Exchange request shape

RFC 8693 defines a standard token exchange request.

A conceptual request:

In [ ]:
token_exchange_request = {
    "grant_type": "urn:ietf:params:oauth:grant-type:token-exchange",
    "subject_token": "<user-access-token>",
    "subject_token_type": "urn:ietf:params:oauth:token-type:access_token",
    "actor_token": "<agent-workload-token>",
    "actor_token_type": "urn:ietf:params:oauth:token-type:access_token",
    "audience": "procurement-api",
    "scope": "purchase_order:create",
}
print(json.dumps(token_exchange_request, indent=2))

In a production design:

```text
Human identity
+
Agent workload identity
   ↓
Authorization server / STS
   ↓
Short-lived task token
```

The resulting token should normally have **less** authority than the human's general account.

## 13. OpenFGA task-based authorization model

In [ ]:
OPENFGA_MODEL = """
model
  schema 1.1

type user

type agent

type department
  relations
    define member: [user]

type task
  relations
    define assignee: [agent]
    define delegator: [user]
    define department: [department]
    define can_read_vendor: assignee
    define can_create_po: assignee

type vendor
  relations
    define approved_for: [department]
"""
print(OPENFGA_MODEL)

Conceptual tuples:

```text
task:task-123#delegator@user:user-123
task:task-123#assignee@agent:procurement-v1
task:task-123#department@department:data-ai
vendor:vendor-acme#approved_for@department:data-ai
```

When the task completes, remove the task tuples to revoke authority.

## 14. Optional OpenFGA Python SDK integration

In [ ]:
try:
    import openfga_sdk
    print("OpenFGA SDK imported.")
except Exception as exc:
    print("OpenFGA SDK import failed:", exc)

FGA_API_URL = os.getenv("FGA_API_URL")
FGA_STORE_ID = os.getenv("FGA_STORE_ID")
FGA_MODEL_ID = os.getenv("FGA_MODEL_ID")

if FGA_API_URL and FGA_STORE_ID:
    print("OpenFGA environment configured.")
    print("Use the installed SDK version's official examples to write tuples and perform checks.")
else:
    print("Optional: set FGA_API_URL, FGA_STORE_ID, and FGA_MODEL_ID for a live OpenFGA lab.")

## 15. Cedar authorization policy example

In [ ]:
CEDAR_POLICY = """
permit (
  principal == Agent::"procurement-v1",
  action == Action::"CreatePurchaseOrder",
  resource
)
when {
  context.task == "task-buy-laptops-001" &&
  context.amount <= 5000 &&
  context.vendorApproved == true
};
"""
print(CEDAR_POLICY)

Cedar and OpenFGA solve related but different problems.

A common architecture could use:

```text
OpenFGA → Is the agent assigned to this task/resource?
Cedar   → Is the action allowed under amount/vendor/context policy?
```

## 16. Confused-deputy test

In [ ]:
g, t = fresh_grant()

confused_request = safe_request.model_copy(update={
    "resource": "department:finance",
    "amount": 2000,
})

authorize(t, confused_request)[:2]

The identity is valid.

The signature is valid.

The task token is valid.

The resource is still unauthorized.

This is the practical difference between **authentication** and **authorization**.

## 17. Audit evidence

In [ ]:
display(pd.DataFrame(AUDIT))

Useful delegation evidence includes:

- human subject,
- actor/agent,
- workload identity,
- task,
- grant ID,
- parent grant,
- audience,
- scope,
- resource,
- constraints,
- authorization result,
- expiry,
- call count,
- execution result.

## 18. Regression tests

In [ ]:
g, t = fresh_grant()
assert authorize(t, safe_request)[0] is True

g, t = fresh_grant()
assert authorize(t, safe_request.model_copy(update={"amount": 5001}))[0] is False

g, t = fresh_grant()
assert authorize(t, safe_request.model_copy(update={"resource": "department:finance"}))[0] is False

try:
    attenuate_grant(grant, research_agent.principal_id, ["payment:issue"])
    raise AssertionError("Privilege amplification should fail.")
except ValueError:
    pass

print("Delegation regression tests passed.")

# 19. Exercises

### A — Expired grant

Create an already-expired grant and verify denial.

### B — Audience confusion

Use a `procurement-api` token against `vendor-api`.

### C — Task completion

Implement:

```python
complete_task(task_id)
```

that revokes all task grants.

### D — Sub-agent

Grant the Research Agent only `vendor:read`.

Verify it cannot create a PO.

### E — Live OpenFGA

Run OpenFGA locally and implement the task tuples using `openfga_sdk`.

### F — Cedar

Model amount/vendor/context authorization in Cedar or Amazon Verified Permissions.

### G — SPIFFE

If you have Kubernetes, deploy SPIRE and replace the simulated workload ID with a real workload identity.

### H — Token Exchange

If your identity provider supports RFC 8693, replace the local token issuer with a standards-based token exchange.

# 20. Key takeaways

1. Human identity, agent identity, and workload identity are distinct.
2. Authentication is not authorization.
3. Delegated authority should preserve the original subject/delegator.
4. Zero standing privilege reduces blast radius.
5. Task-scoped authority should be short-lived and narrow.
6. Audience, resource, amount, and call limits matter.
7. Sub-agent permissions should attenuate.
8. RFC 8693 provides a standard token-exchange pattern.
9. RFC 9700 provides current OAuth security best practice.
10. SPIFFE/SPIRE provide workload identity foundations.
11. OpenFGA is well suited to relationship/task authorization.
12. Cedar is well suited to contextual authorization.
13. Authorization belongs outside model reasoning.
14. Delegation and authorization decisions must be observable and auditable.